# Multi AI Agent System

A **Supervisor Multi-Agent System** using LangGraph where a supervisor LLM routes tasks between a researcher agent and a writer agent.

**Flow:** `START → supervisor → researcher → supervisor → writer → supervisor → END`

## Cell 1: Set API Key

In [13]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
print("Key set (hidden for security).")

Key set (hidden for security).


## Cell 2: Initialise the LLM

In [14]:
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile"
)
print("LLM initialised:", llm.model_name)

LLM initialised: llama-3.3-70b-versatile


## Cell 3: Imports

> **Fix applied:** `create_react_agent` is now imported from `langgraph.prebuilt` (correct location for LangGraph ≥ 1.0). The old import from `langchain.agents` was removed.

In [15]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from pydantic import BaseModel

print("All imports successful.")

All imports successful.


## Cell 4: Define Tools

In [16]:
@tool
def search_web(query: str) -> str:
    """Search the web for information about a topic. Returns simulated results."""
    simulated_results = {
        "python": (
            "Python is a high-level, interpreted programming language created by "
            "Guido van Rossum in 1991. It emphasises readability and supports "
            "multiple programming paradigms."
        ),
        "langgraph": (
            "LangGraph is a library for building stateful, multi-step AI applications "
            "as graphs. It is built on top of LangChain and supports loops, branching, "
            "memory, and multi-agent workflows."
        ),
        "ai agents": (
            "AI agents are autonomous programs that perceive their environment, "
            "make decisions, and take actions to achieve goals. Modern LLM-based "
            "agents use a Think -> Act -> Observe loop."
        ),
    }
    query_lower = query.lower()
    for key, result in simulated_results.items():
        if key in query_lower:
            return f"Search results for '{query}':\n{result}"
    return (
        f"Search results for '{query}':\n"
        "Found general information: This is a topic worth exploring further. "
        "The subject involves multiple interconnected concepts and has grown "
        "significantly in recent years."
    )


@tool
def format_report(content: str, title: str = "Report") -> str:
    """Format raw content into a well-structured report with title and sections."""
    report = (
        f"=== {title.upper()} ===\n\n"
        f"SUMMARY\n"
        f"-------\n"
        f"{content}\n\n"
        f"CONCLUSION\n"
        f"----------\n"
        f"This report was compiled by the Writer agent based on research findings."
    )
    return report


print("Tools defined:")
print(f"  - search_web: {search_web.name}")
print(f"  - format_report: {format_report.name}")

Tools defined:
  - search_web: search_web
  - format_report: format_report


## Cell 5: Create Worker Agents

In [17]:
researcher_agent = create_react_agent(
    model=llm,
    tools=[search_web],
    prompt=(
        "You are a research specialist. "
        "Your job is to find accurate information about any topic. "
        "Use the search_web tool to look up facts. "
        "Return a clear, factual summary of what you found."
    ),
)

writer_agent = create_react_agent(
    model=llm,
    tools=[format_report],
    prompt=(
        "You are a writing specialist. "
        "Your job is to take research findings and turn them into a "
        "well-structured, readable report. "
        "Use the format_report tool to structure the output. "
        "Make the report clear and professional."
    ),
)

print("Worker agents created:")
print("  - researcher_agent")
print("  - writer_agent")

Worker agents created:
  - researcher_agent
  - writer_agent


C:\Users\Admin\AppData\Local\Temp\ipykernel_20628\379475467.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  researcher_agent = create_react_agent(
C:\Users\Admin\AppData\Local\Temp\ipykernel_20628\379475467.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  writer_agent = create_react_agent(


## Cell 6: Define AgentState

In [18]:
class AgentState(MessagesState):
    # 'next' holds the supervisor's routing decision
    next: str

print("AgentState defined with fields: messages, next")

AgentState defined with fields: messages, next


## Cell 7: Define Supervisor Node

In [19]:
class RouteDecision(BaseModel):
    """The routing decision: which agent to call next, or FINISH."""
    next: Literal["researcher", "writer", "FINISH"]

supervisor_llm = llm.with_structured_output(RouteDecision)

SUPERVISOR_SYSTEM_PROMPT = (
    "You are a supervisor managing a team of two specialist agents:\n"
    "\n"
    "  - researcher : Finds facts, searches for information, gathers data\n"
    "  - writer     : Takes research findings and writes a well-formatted report\n"
    "\n"
    "Given the conversation history, decide which agent should act next.\n"
    "\n"
    "Rules:\n"
    "- If the user request involves finding information, route to 'researcher' first\n"
    "- If research has been done and a report is needed, route to 'writer'\n"
    "- If both research and writing are complete, return 'FINISH'\n"
    "- If the task can be answered without research or writing, return 'FINISH'\n"
    "\n"
    "Respond with ONLY the name of the next agent, or 'FINISH'."
)


def supervisor_node(state: AgentState) -> dict:
    """Read the conversation and decide which worker runs next."""
    messages = [SystemMessage(content=SUPERVISOR_SYSTEM_PROMPT)] + state["messages"]
    decision: RouteDecision = supervisor_llm.invoke(messages)
    print(f"  [supervisor] -> routing to: {decision.next}")
    return {"next": decision.next}


print("supervisor_node defined.")

supervisor_node defined.


## Cell 8: Define Worker Nodes

In [20]:
def researcher_node(state: AgentState) -> dict:
    """Run the researcher agent and return its response to the shared state."""
    print("  [researcher] working...")

    # Invoke the researcher agent with the current conversation
    result = researcher_agent.invoke({"messages": state["messages"]})

    # The agent returns a full messages list; we take the last message (its answer)
    last_message = result["messages"][-1]

    # Wrap it in an AIMessage tagged with the agent's name so we can trace it
    agent_message = AIMessage(content=last_message.content, name="researcher")

    # Returning {"messages": [...]} causes MessagesState to APPEND (not replace)
    return {"messages": [agent_message]}


def writer_node(state: AgentState) -> dict:
    """Run the writer agent and return its response to the shared state."""
    print("  [writer] working...")

    # Invoke the writer agent with the current conversation
    result = writer_agent.invoke({"messages": state["messages"]})

    # Take the last message (the writer's output)
    last_message = result["messages"][-1]

    # Tag it with the agent's name
    agent_message = AIMessage(content=last_message.content, name="writer")

    return {"messages": [agent_message]}


print("researcher_node and writer_node defined.")

researcher_node and writer_node defined.


## Cell 9: Build & Compile the Graph

This was the **missing cell** — wires all the nodes together into a runnable `StateGraph`.

In [21]:
# ── Build the graph ──────────────────────────────────────────────
builder = StateGraph(AgentState)

# Register nodes
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer",     writer_node)

# Entry point: always start with the supervisor
builder.add_edge(START, "supervisor")

# Supervisor decides dynamically which node runs next
builder.add_conditional_edges(
    "supervisor",
    lambda state: state["next"],   # reads the 'next' field set by supervisor_node
    {
        "researcher": "researcher",
        "writer":     "writer",
        "FINISH":     END,
    }
)

# After each worker finishes, hand control back to the supervisor
builder.add_edge("researcher", "supervisor")
builder.add_edge("writer",     "supervisor")

# Compile into a runnable graph
graph = builder.compile()

print("Graph compiled successfully!")
print("Nodes:", list(builder.nodes.keys()))

Graph compiled successfully!
Nodes: ['supervisor', 'researcher', 'writer']


## Cell 10: Run the Graph

In [22]:
print("=" * 60)
print("TASK: Research LangGraph and write a report about it")
print("=" * 60)

initial_input = {
    "messages": [HumanMessage(content=(
        "Please research LangGraph and then write a short report about it. "
        "First gather the facts, then format them into a report."
    ))]
}

final_state = graph.invoke(initial_input)

print()
print("=" * 60)
print("FINAL OUTPUT")
print("=" * 60)

for i, msg in enumerate(final_state["messages"]):
    role = getattr(msg, "name", None) or type(msg).__name__
    print(f"\n[{i+1}] {role.upper()}")
    print("-" * 40)
    print(msg.content[:600])
    if len(msg.content) > 600:
        print("  ... (truncated)")

TASK: Research LangGraph and write a report about it
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...
  [supervisor] -> routing to: researcher
  [researcher] working...


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_web,{"query": "LangGraph developers researchers AI"}</function>'}}